# Day 16 — The ReAct pattern

**ReAct = Reasoning + Acting.** The agent interleaves a **Thought** (private reasoning),
an **Action** (a tool call), and an **Observation** (the result) — over and over until it
can `finish`. It's the most influential agent pattern, and you can build it with plain
`chat()`.

We make the model emit **JSON** each turn so parsing is reliable:

```json
{"thought": "I should multiply", "action": "calculator", "action_input": "25 * 4"}
```

Compare with Day 15: there the *provider* structured the call; here *you* do, which works
with any model and makes the reasoning visible.

In [2]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

repo root on sys.path: c:\autogen\AILearning


## 🔬 Your turn

Fill in the `TODO`s, then run the cell. Stuck? The solution is below — but try first.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator, clock

TOOLS = {"calculator": calculator, "clock": clock}
SYSTEM = (
    "You solve tasks by reasoning, then acting. Reply EACH turn with ONLY a JSON object:\n"
    '{"thought": "...", "action": "calculator|clock|finish", "action_input": "..."}\n'
    "- calculator: action_input is an arithmetic expression\n"
    "- clock: action_input is ignored\n"
    "- finish: action_input is your final answer to the user"
)

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def run(goal, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        obj = parse_json(chat(messages, temperature=0))
        # TODO 1: if verbose, print obj["thought"]
        # TODO 2: if obj["action"] == "finish": return obj["action_input"]
        # TODO 3: else run TOOLS[action](action_input); append assistant + a
        #         user message "Observation: <result>" and continue the loop
        raise NotImplementedError("complete the ReAct loop")
    return "stopped (hit max_steps)"

# print(run("What is 25 * 4, and what time is it now?"))

## 🔒 Solution

One correct way to do it. Compare it with your version.

In [3]:
import json
from shared.llm import chat
from shared.tools import calculator, clock

TOOLS = {"calculator": calculator, "clock": clock}
SYSTEM = (
    "You solve tasks by reasoning, then acting. Reply EACH turn with ONLY a JSON object:\n"
    '{"thought": "...", "action": "calculator|clock|finish", "action_input": "..."}\n'
    "- calculator: action_input is an arithmetic expression\n"
    "- clock: action_input is ignored\n"
    "- finish: action_input is your final answer to the user"
)

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def run(goal, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        raw = chat(messages, temperature=0)
        obj = parse_json(raw)
        if verbose:
            print(f"[{step}] 💭 {obj.get('thought', '')}")
        if obj.get("action") == "finish":
            return obj.get("action_input", "")
        tool = TOOLS.get(obj.get("action"))
        obs = tool(obj.get("action_input", "")) if tool else f"unknown tool: {obj.get('action')}"
        if verbose:
            print(f"     🔧 {obj.get('action')}({obj.get('action_input', '')!r}) -> {obs}")
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "stopped (hit max_steps)"

print(run("What is 25 * 4, and what time is it now?"))

[1] 💭 First, I need to calculate the product of 25 and 4. Then, I should check the current time.
     🔧 calculator('25*4') -> 100
[2] 💭 Now that I have the result of the multiplication, I can proceed to find out the current time.
     🔧 clock('') -> 2026-06-20 19:00:00
[3] 💭 I've obtained the current time. Now, I should provide a final answer to the user's original question.
100
